In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import *
from modules.CliffsDelta import cliffsDelta

In [2]:
noise_free_dir = f"./data/NoiseFreeDataset/"
noisy_dir = f"./data/NoisyDataset/"

cols = ["SLOCStandard", "Readability", "McCabe", "totalFanOut", "MaintainabilityIndex"]

# Wilcoxon rank sum test and Cliff's delta

## Noisy NotBuggy Vs Noisy Buggy

In [7]:
results_wrst = []
results_cd = []

all_files = [f for f in os.listdir(noise_free_dir) if f.endswith(".csv")]

method_count = []

for file in all_files:
    df = pd.read_csv(os.path.join(noisy_dir, file))

    noisy_buggy = df[df["Decision"] == "Buggy"][cols].copy()
    noisy_notbuggy = df[df["Decision"] == "NotBuggy"][cols].copy()


    res_wrst = {"Project": file.replace('.csv', '')}
    res_wrst["#Methods"] = len(noisy_buggy) + len(noisy_notbuggy)

    for col in cols:
        z_stat, p_value = ranksums(noisy_buggy[col], noisy_notbuggy[col])
        is_diff = p_value < 0.05
        res_wrst[col] = is_diff
    results_wrst.append(res_wrst)

    res_cd = {"Project": file.replace('.csv', '')}
    res_cd["#Methods"] = len(noisy_buggy) + len(noisy_notbuggy)
    for col in cols:
        if noisy_buggy.shape[0] == 0 or noisy_notbuggy.shape[0] == 0:
            size = "-"
        else:
            d, size = cliffsDelta(noisy_buggy[col], noisy_notbuggy[col])
        res_cd[col] = size
    results_cd.append(res_cd)

wrst_results_df = pd.DataFrame(results_wrst)

true_percentages_all = wrst_results_df.drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_all = true_percentages_all.reset_index()
true_percentages_df_all.columns = ['column', 'All']
true_percentages_df_all = true_percentages_df_all.set_index('column').T

true_percentages_t20 = wrst_results_df.sort_values(by=["#Methods"], ascending=False).head(20).drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_t20 = true_percentages_t20.reset_index()
true_percentages_df_t20.columns = ['column', 'Top 20']
true_percentages_df_t20 = true_percentages_df_t20.set_index('column').T

wrst = pd.concat([true_percentages_df_all, true_percentages_df_t20]).round(2)



cd_results_df = pd.DataFrame(results_cd)

# All projects
res = []
for col in cols:
    percentage = pd.DataFrame(cd_results_df[col].value_counts(normalize=True) * 100)
    res.append({
        'column': col,
        'negligible': percentage.loc["negligible"]["proportion"] if "negligible" in percentage.index else 0,
        'small': percentage.loc["small"]["proportion"] if "small" in percentage.index else 0,
        'medium': percentage.loc["medium"]["proportion"] if "medium" in percentage.index else 0,
        'large': percentage.loc["large"]["proportion"] if "large" in percentage.index else 0,
    })
cd = pd.DataFrame(res).round(2)

/tmp/ipykernel_972619/1894392411.py:19: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  z_stat, p_value = ranksums(noisy_buggy[col], noisy_notbuggy[col])


In [8]:
wrst

column,SLOCStandard,Readability,McCabe,totalFanOut,MaintainabilityIndex
All,95.92,91.84,95.92,97.96,95.92
Top 20,100.00,100.00,100.00,100.00,100.00


In [9]:
cd

,column,negligible,small,medium,large
0,SLOCStandard,0.00,14.29,46.94,36.73
1,Readability,2.04,36.73,42.86,16.33
2,McCabe,2.04,34.69,36.73,24.49
3,totalFanOut,0.00,10.20,46.94,40.82
4,MaintainabilityIndex,0.00,12.24,34.69,51.02


## Noise-Free NotBuggy vs Noisy Buggy

In [3]:
results_wrst = []
results_cd = []

all_files = [f for f in os.listdir(noise_free_dir) if f.endswith(".csv")]

method_count = []

for file in all_files:
    df = pd.read_csv(os.path.join(noisy_dir, file))
    noisy_buggy = df[df["Decision"] == "Buggy"][cols].copy()

    df = pd.read_csv(os.path.join(noise_free_dir, file))
    noise_free_notbuggy = df[df["Detection"] == "NotBuggy"][cols].copy()


    res_wrst = {"Project": file.replace('.csv', '')}
    res_wrst["#Methods"] = len(noisy_buggy) + len(noise_free_notbuggy)

    for col in cols:
        z_stat, p_value = ranksums(noisy_buggy[col], noise_free_notbuggy[col])
        is_diff = p_value < 0.05
        res_wrst[col] = is_diff
    results_wrst.append(res_wrst)

    res_cd = {"Project": file.replace('.csv', '')}
    res_cd["#Methods"] = len(noisy_buggy) + len(noise_free_notbuggy)
    for col in cols:
        d, size = cliffsDelta(noisy_buggy[col], noise_free_notbuggy[col])
        res_cd[col] = size
    results_cd.append(res_cd)

wrst_results_df = pd.DataFrame(results_wrst)

true_percentages_all = wrst_results_df.drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_all = true_percentages_all.reset_index()
true_percentages_df_all.columns = ['column', 'All']
true_percentages_df_all = true_percentages_df_all.set_index('column').T

true_percentages_t20 = wrst_results_df.sort_values(by=["#Methods"], ascending=False).head(20).drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_t20 = true_percentages_t20.reset_index()
true_percentages_df_t20.columns = ['column', 'Top 20']
true_percentages_df_t20 = true_percentages_df_t20.set_index('column').T

wrst = pd.concat([true_percentages_df_all, true_percentages_df_t20]).round(2)



cd_results_df = pd.DataFrame(results_cd)

# All projects
res = []
for col in cols:
    percentage = pd.DataFrame(cd_results_df[col].value_counts(normalize=True) * 100)
    res.append({
        'column': col,
        'negligible': percentage.loc["negligible"]["proportion"] if "negligible" in percentage.index else 0,
        'small': percentage.loc["small"]["proportion"] if "small" in percentage.index else 0,
        'medium': percentage.loc["medium"]["proportion"] if "medium" in percentage.index else 0,
        'large': percentage.loc["large"]["proportion"] if "large" in percentage.index else 0,
    })
cd = pd.DataFrame(res).round(2)

wrst

column,SLOCStandard,Readability,McCabe,totalFanOut,MaintainabilityIndex
All,97.96,93.88,97.96,100.0,97.96
Top 20,100.00,100.00,100.00,100.0,100.00


In [4]:
cd

,column,negligible,small,medium,large
0,SLOCStandard,0.00,16.33,48.98,34.69
1,Readability,4.08,36.73,42.86,16.33
2,McCabe,2.04,40.82,32.65,24.49
3,totalFanOut,2.04,12.24,46.94,38.78
4,MaintainabilityIndex,0.00,14.29,44.90,40.82


## Noisy NotBuggy vs Noise-Free Buggy

In [3]:
results_wrst = []
results_cd = []

all_files = [f for f in os.listdir(noise_free_dir) if f.endswith(".csv")]

method_count = []

for file in all_files:
    df = pd.read_csv(os.path.join(noisy_dir, file))
    noisy_notbuggy = df[df["Decision"] == "NotBuggy"][cols].copy()

    noise_free_buggy = pd.read_csv(os.path.join(noise_free_dir, file))
    noise_free_buggy = noise_free_buggy[noise_free_buggy["Detection"] == "Buggy"][cols].copy()

    res_wrst = {"Project": file.replace('.csv', '')}
    res_wrst["#Methods"] = len(noise_free_buggy) + len(noisy_notbuggy)

    for col in cols:
        z_stat, p_value = ranksums(noise_free_buggy[col], noisy_notbuggy[col])
        is_diff = p_value < 0.05
        res_wrst[col] = is_diff
    results_wrst.append(res_wrst)

    res_cd = {"Project": file.replace('.csv', '')}
    res_cd["#Methods"] = len(noise_free_buggy) + len(noisy_notbuggy)
    for col in cols:
        if noise_free_buggy.shape[0] == 0 or noisy_notbuggy.shape[0] == 0:
            size = "-"
        else:
            d, size = cliffsDelta(noise_free_buggy[col], noisy_notbuggy[col])
        res_cd[col] = size
    results_cd.append(res_cd)

wrst_results_df = pd.DataFrame(results_wrst)

true_percentages_all = wrst_results_df.drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_all = true_percentages_all.reset_index()
true_percentages_df_all.columns = ['column', 'All']
true_percentages_df_all = true_percentages_df_all.set_index('column').T

true_percentages_t20 = wrst_results_df.sort_values(by=["#Methods"], ascending=False).head(20).drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_t20 = true_percentages_t20.reset_index()
true_percentages_df_t20.columns = ['column', 'Top 20']
true_percentages_df_t20 = true_percentages_df_t20.set_index('column').T

wrst = pd.concat([true_percentages_df_all, true_percentages_df_t20]).round(2)


cd_results_df = pd.DataFrame(results_cd)

# All projects
res = []
for col in cols:
    percentage = pd.DataFrame(cd_results_df[col].value_counts(normalize=True) * 100)
    res.append({
        'column': col,
        'negligible': percentage.loc["negligible"]["proportion"] if "negligible" in percentage.index else 0,
        'small': percentage.loc["small"]["proportion"] if "small" in percentage.index else 0,
        'medium': percentage.loc["medium"]["proportion"] if "medium" in percentage.index else 0,
        'large': percentage.loc["large"]["proportion"] if "large" in percentage.index else 0,
    })
cd = pd.DataFrame(res).round(2)

wrst

/tmp/ipykernel_976467/3911123787.py:19: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  z_stat, p_value = ranksums(noise_free_buggy[col], noisy_notbuggy[col])


column,SLOCStandard,Readability,McCabe,totalFanOut,MaintainabilityIndex
All,95.92,91.84,95.92,97.96,95.92
Top 20,100.00,100.00,100.00,100.00,100.00


In [4]:
cd

,column,negligible,small,medium,large
0,SLOCStandard,0.00,6.12,18.37,73.47
1,Readability,2.04,22.45,38.78,34.69
2,McCabe,2.04,6.12,40.82,48.98
3,totalFanOut,0.00,4.08,18.37,75.51
4,MaintainabilityIndex,0.00,6.12,10.20,81.63


## Noise-Free NotBuggy vs Noise-Free Buggy

In [3]:
results_wrst = []
results_cd = []

all_files = [f for f in os.listdir(noise_free_dir) if f.endswith(".csv")]

method_count = []

for file in all_files:
    df = pd.read_csv(os.path.join(noise_free_dir, file))

    noise_free_buggy = df[df["Detection"] == "Buggy"][cols].copy()
    noise_free_notbuggy = df[df["Detection"] == "NotBuggy"][cols].copy()

    res_wrst = {"Project": file.replace('.csv', '')}
    res_wrst["#Methods"] = len(noise_free_buggy) + len(noise_free_notbuggy)

    for col in cols:
        z_stat, p_value = ranksums(noise_free_buggy[col], noise_free_notbuggy[col])
        is_diff = p_value < 0.05
        res_wrst[col] = is_diff
    results_wrst.append(res_wrst)

    res_cd = {"Project": file.replace('.csv', '')}
    res_cd["#Methods"] = len(noise_free_buggy) + len(noise_free_notbuggy)
    for col in cols:
        if noise_free_buggy.shape[0] == 0 or noise_free_notbuggy.shape[0] == 0:
            size = "-"
        else:
            d, size = cliffsDelta(noise_free_buggy[col], noise_free_notbuggy[col])
        res_cd[col] = size
    results_cd.append(res_cd)

wrst_results_df = pd.DataFrame(results_wrst)

true_percentages_all = wrst_results_df.drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_all = true_percentages_all.reset_index()
true_percentages_df_all.columns = ['column', 'All']
true_percentages_df_all = true_percentages_df_all.set_index('column').T

true_percentages_t20 = wrst_results_df.sort_values(by=["#Methods"], ascending=False).head(20).drop(["Project", "#Methods"], axis=1).mean() * 100
true_percentages_df_t20 = true_percentages_t20.reset_index()
true_percentages_df_t20.columns = ['column', 'Top 20']
true_percentages_df_t20 = true_percentages_df_t20.set_index('column').T

wrst = pd.concat([true_percentages_df_all, true_percentages_df_t20]).round(2)


cd_results_df = pd.DataFrame(results_cd)

# All projects
res = []
for col in cols:
    percentage = pd.DataFrame(cd_results_df[col].value_counts(normalize=True) * 100)
    res.append({
        'column': col,
        'negligible': percentage.loc["negligible"]["proportion"] if "negligible" in percentage.index else 0,
        'small': percentage.loc["small"]["proportion"] if "small" in percentage.index else 0,
        'medium': percentage.loc["medium"]["proportion"] if "medium" in percentage.index else 0,
        'large': percentage.loc["large"]["proportion"] if "large" in percentage.index else 0,
    })
cd = pd.DataFrame(res).round(2)

wrst

column,SLOCStandard,Readability,McCabe,totalFanOut,MaintainabilityIndex
All,97.96,93.88,97.96,100.0,97.96
Top 20,100.00,100.00,100.00,100.0,100.00


In [4]:
cd

,column,negligible,small,medium,large
0,SLOCStandard,0.00,6.12,20.41,73.47
1,Readability,2.04,24.49,40.82,32.65
2,McCabe,2.04,8.16,42.86,46.94
3,totalFanOut,0.00,6.12,22.45,71.43
4,MaintainabilityIndex,0.00,6.12,16.33,77.55
